# Track Optimization Notebook (Adam Optimizer)
## Four-stage optimization: hierarchical grid position search + hierarchical cone direction search + energy scan + Adam gradient descent

This notebook implements a state-of-the-art optimization approach:
- **Stage 1**: Hierarchical grid search for optimal origin position using origin_time_loss
- **Stage 2**: Hierarchical cone-based direction search with adaptive refinement
- **Stage 3**: Energy scan around initial guess based on total observed photon counts
- **Stage 4**: Combined loss Adam optimization (position + direction + t0 + energy)

In [ ]:
import sys
sys.path.append('..')

import jax
import jax.numpy as jnp
import time
import math
import numpy as np
from functools import partial
import pickle
from tqdm import tqdm
from jax import grad, jit, vmap, value_and_grad
import uproot
from jax import jit
from pathlib import Path
import optax  # JAX optimization library

from matplotlib import pyplot as plt
plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10

from tools.geometry import generate_detector
from tools.utils import load_single_event, save_single_event, generate_random_params, print_particle_params
from tools.simulation import setup_event_simulator
from tools.generate import read_photon_data_from_photonsim

from tools.optimization.utils.functions import estimate_muon_energy_from_photon_count, cone_points
from tools.optimization.utils.functions import hierarchical_direction_search_cone, energy_scan_optimization
from tools.optimization.utils.functions import cartesian_to_spherical, spherical_to_cartesian

In [ ]:
# Load configuration from file
from tools.optimization.optimize import load_optimization_config, print_optimization_parameters, get_detector_params_from_config
from tools.optimization.optimize import get_detector_bounds, hierarchical_position_grid_search

config = load_optimization_config('../config/single_ring_optimization_config.json')

# Extract basic configuration
default_json_filename = config['basic_config']['default_json_filename']
data_file = config['basic_config']['data_file']
TEMPERATURE = config['basic_config']['temperature']
N_EVENTS = config['basic_config']['n_events']
K = config['basic_config']['k']
Nphot = config['basic_config']['nphot']
C_MEDIUM = config['basic_config']['c_medium']

# Extract optimization weights
VERTEX_WEIGHT_SCALE = config['optimization_weights']['vertex_weight_scale']
COUNTS_WEIGHT_SCALE = config['optimization_weights']['counts_weight_scale']
ENERGY_WEIGHT_SCALE = config['optimization_weights']['energy_weight_scale']

# Extract learning rates - we'll use these to configure Adam
POSITION_LEARNING_RATE = config['learning_rates']['position_learning_rate']
DIRECTION_LEARNING_RATE = config['learning_rates']['direction_learning_rate']
T0_LEARNING_RATE = config['learning_rates']['t0_learning_rate']
ENERGY_LEARNING_RATE = config['learning_rates']['energy_learning_rate']

# Adam-specific parameters (can be added to config later)
ADAM_LEARNING_RATE = 0.2  # Base learning rate for Adam
ADAM_B1 = 0.9  # Exponential decay rate for first moment
ADAM_B2 = 0.999  # Exponential decay rate for second moment
ADAM_EPS = 1e-8  # Small constant for numerical stability

# Extract position+t0 grid search parameters
POS_N_DIV = config['position_grid_search']['pos_n_div']
POS_LEVELS = config['position_grid_search']['pos_levels']
POS_FRACTION = config['position_grid_search']['pos_fraction']
POS_MIN_L = config['position_grid_search']['pos_min_L']
T0_N_DIV = config['position_grid_search']['t0_n_div']
T0_MIN = config['position_grid_search']['t0_min']
T0_MAX = config['position_grid_search']['t0_max']

# Extract cone direction search parameters
CONE_LEVELS = config['cone_direction_search']['cone_levels']
CONE_INITIAL_DIV = config['cone_direction_search']['cone_initial_div']
CONE_MAX_ANGLE_DEG = config['cone_direction_search']['cone_max_angle_deg']
CONE_REDUCTION = config['cone_direction_search']['cone_reduction']

# Extract energy optimization parameters
ENERGY_DELTA = config['energy_optimization']['energy_delta']
ENERGY_SCAN_STEPS = config['energy_optimization']['energy_scan_steps']

# Extract gradient descent parameters
MAX_ITERATIONS = config['gradient_descent']['max_iterations']

# Extract visualization parameters
ARROW_EVERY_N_POINTS = config['visualization']['arrow_every_n_points']

# Setup detector
detector = generate_detector(default_json_filename)
detector_points = jnp.array(detector.all_points)
detector_radius = detector.S_radius
NUM_DETECTORS = len(detector_points)

# Get detector bounds for the new grid search method
detector_bounds = get_detector_bounds(detector)
DETECTOR_R = detector_bounds['r'] if detector_bounds['type'] == 'cylinder' else None
DETECTOR_H = detector_bounds['H'] if detector_bounds['type'] == 'cylinder' else None

# Setup prediction simulator with fixed temperature (is_data=False)
prediction_simulator = setup_event_simulator(default_json_filename, Nphot, TEMPERATURE, max_sensors_per_cell=8, K=K, is_data=False)

# Setup data simulator for generating target events (is_data=True, temperature=0.0)
data_simulator = setup_event_simulator(default_json_filename, Nphot, temperature=0.0, K=K,
                                      is_data=True, is_calibration=False)

# Print parameters using the new function
print_optimization_parameters(config, detector_bounds['r'], detector_bounds.get('H', 0), NUM_DETECTORS)
print(f"\nAdam Optimizer Parameters:")
print(f"  Learning rate: {ADAM_LEARNING_RATE}")
print(f"  Beta1: {ADAM_B1}")
print(f"  Beta2: {ADAM_B2}")
print(f"  Epsilon: {ADAM_EPS}")

# Load ROOT file information
with uproot.open(data_file) as file:
    tree = file['OpticalPhotons']
    n_entries = tree.num_entries
print(f"ROOT file has {n_entries} entries")

# Get detector parameters from config
detector_params = get_detector_params_from_config(config)

In [ ]:
from tools.optimization.losses import energy_loss, counts_loss, origin_time_loss

@jit
def combined_product_loss(params, hit_detector_positions, observed_times, observed_counts, 
                                    true_data, detector_params, key, 
                                    vertex_weight=VERTEX_WEIGHT_SCALE, counts_weight=COUNTS_WEIGHT_SCALE, 
                                    energy_weight=ENERGY_WEIGHT_SCALE):
    """
    Combined loss function: product of vertex loss, counts loss, and energy loss
    
    Args:
        params: [x, y, z, t0, theta, phi, energy] where theta and phi are spherical direction angles
        vertex_weight: scaling factor for vertex loss contribution
        counts_weight: scaling factor for counts loss contribution  
        energy_weight: scaling factor for energy loss contribution
    """
    position = params[:3]
    t0 = params[3]
    theta = params[4]
    phi = params[5]
    energy = params[6]

    track_params = (energy, position, jnp.array([theta, phi]))
    simulated_data = prediction_simulator(track_params, detector_params, key)
    simulated_counts = simulated_data[0]
    simulated_time = simulated_data[1]
    
    # Calculate individual loss components
    vertex_loss_val = origin_time_loss(position, hit_detector_positions, observed_times, observed_counts, t0, scale_w=5000)
    counts_loss_val = counts_loss(observed_counts, simulated_counts)
    energy_loss_val = energy_loss(simulated_counts, observed_counts)
    
    # Weighted product loss with small offset to avoid zero
    combined_loss = jnp.sqrt((vertex_weight * vertex_loss_val + 1e-6) * (counts_weight * counts_loss_val + 1e-6))
    
    return combined_loss, (vertex_loss_val, counts_loss_val, energy_loss_val)

combined_grad_fn = jit(value_and_grad(combined_product_loss, has_aux=True))

In [ ]:
def run_complete_optimization_adam(initial_t0, hit_detector_positions, observed_times, observed_counts,
                                   true_data, true_energy, true_position, true_direction, TRUE_T0,
                                   adam_lr=ADAM_LEARNING_RATE,
                                   vertex_weight=VERTEX_WEIGHT_SCALE, counts_weight=COUNTS_WEIGHT_SCALE, 
                                   energy_weight=ENERGY_WEIGHT_SCALE,
                                   max_iterations=3000, tolerance=1e-6, verbosity=None):
    """
    Run complete optimization pipeline with Adam optimizer:
    1. Energy estimation from photon count
    2. Position+t0 grid search (t0-loop approach: run 3D search for each t0)
    3. Hierarchical cone direction search
    4. Energy scan optimization
    5. Adam optimization (position + direction + t0 + energy)
    
    Args:
        initial_t0: initial t0 guess (used as center for t0 grid)
        adam_lr: Adam optimizer learning rate
        vertex_weight, counts_weight, energy_weight: scaling factors for loss components
        verbosity: Verbosity level (0, 1, 2). If None, uses config value.
    """
    
    # Get verbosity from config if not provided
    if verbosity is None:
        verbosity = config.get('verbosity', {}).get('level', 2)
    
    # Stage 0: Energy estimation from photon count
    if verbosity >= 2:
        print("  Stage 0: Energy estimation from photon count")
    N_photons = jnp.sum(observed_counts)
    energy_guess = estimate_muon_energy_from_photon_count(N_photons)
    if verbosity >= 2:
        print(f"    Observed photons: {N_photons}")
        print(f"    Energy guess: {energy_guess:.1f} (true: {true_energy:.1f})")
    
    # Stage 1: Position+t0 grid search using t0-loop approach
    if verbosity >= 2:
        print("  Stage 1: Position+t0 grid search (t0-loop approach)")
    pos_results = hierarchical_position_grid_search(
        hit_detector_positions, observed_times, observed_counts,
        true_position, TRUE_T0, initial_t0, detector_bounds,
        n_div=POS_N_DIV, t0_n_div=T0_N_DIV, levels=POS_LEVELS, fraction=POS_FRACTION,
        t0_min=T0_MIN, t0_max=T0_MAX,
        min_L=POS_MIN_L, verbosity=verbosity)
    
    optimal_position = pos_results['best_position']
    optimal_t0 = pos_results['best_t0']
    
    if optimal_position is None or optimal_t0 is None:
        if verbosity >= 1:
            print("  ERROR: Position+t0 search failed to find valid position")
        return None
    
    # Stage 2: Hierarchical cone direction search at optimal position and t0
    if verbosity >= 2:
        print("  Stage 2: Hierarchical cone direction search at optimal position")
    cone_results = hierarchical_direction_search_cone(
        prediction_simulator, detector_params, optimal_position, optimal_t0, hit_detector_positions, observed_times, observed_counts,
        true_data, energy_guess, levels=CONE_LEVELS, initial_div=CONE_INITIAL_DIV, max_angle_deg=CONE_MAX_ANGLE_DEG, reduction=CONE_REDUCTION, verbosity=verbosity
    )
    
    # Stage 3: Energy scan optimization
    if verbosity >= 2:
        print("  Stage 3: Energy scan optimization")
    energy_scan_results = energy_scan_optimization(
        prediction_simulator, detector_params, optimal_position, cone_results['best_theta'], cone_results['best_phi'],
        optimal_t0, hit_detector_positions, observed_times, observed_counts,
        true_data, energy_guess, energy_delta=ENERGY_DELTA, n_steps=ENERGY_SCAN_STEPS, verbosity=verbosity
    )
    
    optimal_energy = energy_scan_results['best_energy']
    
    # Stage 4: Adam optimization with parameter-specific update scaling
    if verbosity >= 2:
        print("  Stage 4: Adam optimization (position + direction + t0 + energy)")
        print("    Update scaling: position=1.0, direction=0.2 (1/5), t0=1.0, energy=10.0")
    
    # Create initial parameter vector with all optimized components
    initial_params = jnp.array([
        optimal_position[0], optimal_position[1], optimal_position[2],
        optimal_t0,
        cone_results['best_theta'], cone_results['best_phi'],
        optimal_energy
    ])
    
    # Initialize Adam optimizer
    optimizer = optax.adam(learning_rate=adam_lr, b1=ADAM_B1, b2=ADAM_B2, eps=ADAM_EPS)
    opt_state = optimizer.init(initial_params)
    
    # Define parameter-specific scaling factors
    # Parameter structure: [x, y, z, t0, theta, phi, energy]
    position_scale = 1.0    
    t0_scale = 10.0          
    direction_scale = 0.025  
    energy_scale = 10.0     
    
    update_scales = jnp.array([
        position_scale,  # x
        position_scale,  # y
        position_scale,  # z
        t0_scale,        # t0
        direction_scale, # theta
        direction_scale, # phi
        energy_scale     # energy
    ])
    
    current_params = initial_params.copy()
    
    history = {
        'parameters': [current_params.copy()],
        'combined_losses': [],
        'vertex_losses': [],
        'counts_losses': [],
        'energy_losses': [],
        'position_errors': [],
        'direction_errors': [],
        't0_errors': [],
        'energy_errors': []
    }
    
    # Random key for counts loss evaluations
    opt_key = jax.random.PRNGKey(12345)
    
    if verbosity >= 2:
        print(f"    Starting Adam optimization...")

    dumping_factor = 0.998
    current_dumping_w = 1.
    for iteration in range(max_iterations):
        opt_key, _ = jax.random.split(opt_key)
        
        (combined_loss, (vertex_loss, counts_loss_val, energy_loss_val)), grad = combined_grad_fn(
            current_params, hit_detector_positions, observed_times, observed_counts,
            true_data, detector_params, opt_key, vertex_weight=vertex_weight, counts_weight=counts_weight, energy_weight=energy_weight
        )
        
        grad_norm = jnp.linalg.norm(grad)
        if grad_norm < tolerance:
            break
        
        # Adam update with parameter-specific scaling
        updates, opt_state = optimizer.update(grad, opt_state, current_params)

        current_dumping_w *= dumping_factor
        #print(current_dumping_w)
        # Apply parameter-specific scaling to updates
        scaled_updates = updates * update_scales * current_dumping_w
        
        # Apply scaled updates to parameters
        current_params = optax.apply_updates(current_params, scaled_updates)
        
        # Apply constraints based on detector type
        if DETECTOR_R is not None and DETECTOR_H is not None:
            # Cylinder detector
            current_params = jnp.array([
                jnp.clip(current_params[0], -DETECTOR_R * 0.9, DETECTOR_R * 0.9),  # x
                jnp.clip(current_params[1], -DETECTOR_R * 0.9, DETECTOR_R * 0.9),  # y
                jnp.clip(current_params[2], -DETECTOR_H/2 * 0.9, DETECTOR_H/2 * 0.9),  # z
                jnp.clip(current_params[3], -20.0, 20.0),  # t0
                current_params[4],  # theta
                current_params[5],  # phi
                jnp.clip(current_params[6], 100.0, 2000.0)  # energy (reasonable bounds)
            ])
        else:
            # Generic constraints for other detector types
            current_params = jnp.array([
                current_params[0],  # x
                current_params[1],  # y
                current_params[2],  # z
                jnp.clip(current_params[3], -20.0, 20.0),  # t0
                current_params[4],  # theta
                current_params[5],  # phi
                jnp.clip(current_params[6], 100.0, 2000.0)  # energy
            ])
        
        # Calculate current errors for tracking
        current_position = current_params[:3]
        current_t0 = current_params[3]
        current_theta = current_params[4]
        current_phi = current_params[5]
        current_energy = current_params[6]
        current_direction = spherical_to_cartesian(current_theta, current_phi)
        
        position_error = jnp.linalg.norm(current_position - true_position)
        t0_error = abs(current_t0 - TRUE_T0)
        energy_error = abs(current_energy - true_energy)
        cos_angle = jnp.clip(jnp.dot(current_direction, true_direction), -1.0, 1.0)
        direction_error = np.degrees(np.arccos(cos_angle))

        if verbosity >= 2 and ((iteration+1) % 100 == 0 or iteration == 0):
            print(f"      Iter {iteration}: pos_err={position_error:.3f}m, t0_err={t0_error:.3f}, dir_err={direction_error:.3f}°, E_err={energy_error:.1f}")
            print(f"      Loss: {combined_loss:.6f}, grad_norm: {grad_norm:.6f}")
        
        # Store history
        history['parameters'].append(current_params.copy())
        history['combined_losses'].append(float(combined_loss))
        history['vertex_losses'].append(float(vertex_loss))
        history['counts_losses'].append(float(counts_loss_val))
        history['energy_losses'].append(float(energy_loss_val))
        history['position_errors'].append(float(position_error))
        history['direction_errors'].append(float(direction_error))
        history['t0_errors'].append(float(t0_error))
        history['energy_errors'].append(float(energy_error))
    
    # Final calculations
    final_position = current_params[:3]
    final_t0 = current_params[3]
    final_theta = current_params[4]
    final_phi = current_params[5]
    final_energy = current_params[6]
    final_direction = spherical_to_cartesian(final_theta, final_phi)
    
    final_position_error = jnp.linalg.norm(final_position - true_position)
    final_t0_error = abs(final_t0 - TRUE_T0)
    final_energy_error = abs(final_energy - true_energy)
    final_cos_angle = jnp.clip(jnp.dot(final_direction, true_direction), -1.0, 1.0)
    final_direction_error = np.degrees(np.arccos(final_cos_angle))
    
    return {
        'energy_estimation': {
            'n_photons': N_photons,
            'energy_guess': float(energy_guess),
            'energy_guess_error': float(abs(energy_guess - true_energy))
        },
        'grid_position_search': pos_results,
        'cone_direction_search': cone_results,
        'energy_scan_search': energy_scan_results,
        'initial_params': initial_params,
        'final_position': final_position,
        'final_direction': final_direction,
        'final_theta': final_theta,
        'final_phi': final_phi,
        'final_t0': final_t0,
        'final_energy': final_energy,
        'final_combined_loss': history['combined_losses'][-1] if history['combined_losses'] else float('inf'),
        'final_vertex_loss': history['vertex_losses'][-1] if history['vertex_losses'] else float('inf'),
        'final_counts_loss': history['counts_losses'][-1] if history['counts_losses'] else float('inf'),
        'final_energy_loss': history['energy_losses'][-1] if history['energy_losses'] else float('inf'),
        'final_position_error': float(final_position_error),
        'final_direction_error': float(final_direction_error),
        'final_t0_error': float(final_t0_error),
        'final_energy_error': float(final_energy_error),
        'total_iterations': len(history['parameters']) - 1,
        'converged': grad_norm < tolerance,
        'history': history
    }

print("Adam optimization function defined")

In [ ]:
def generate_event_data(event_idx, random_key):
    """
    Generate a single event with random parameters within detector bounds
    """
    entry_idx = event_idx % n_entries
    
    photon_data = read_photon_data_from_photonsim(data_file, entry_idx)
    photon_data['N'] = len(photon_data['photon_origins'])
    
    key = random_key
    fraction = 0.6
    
    r_vert = jax.random.uniform(key, shape=(), minval=0, maxval=DETECTOR_R * fraction)
    key, _ = jax.random.split(key)
    theta = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
    key, _ = jax.random.split(key)
    z_vert = jax.random.uniform(key, shape=(), minval=-DETECTOR_H/2 * fraction, 
                               maxval=DETECTOR_H/2 * fraction)
    true_position = jnp.array([r_vert * jnp.cos(theta), r_vert * jnp.sin(theta), z_vert])
    
    key, _ = jax.random.split(key)
    phi = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
    key, _ = jax.random.split(key)
    cos_theta = jax.random.uniform(key, shape=(), minval=-1, maxval=1)
    sin_theta = jnp.sqrt(1 - cos_theta**2)
    true_direction = jnp.array([sin_theta * jnp.cos(phi), sin_theta * jnp.sin(phi), cos_theta])
    
    true_energy = photon_data['energy']
    TRUE_T0 = jax.random.uniform(key, shape=(), minval=-3.0, maxval=3.0)
    
    true_params = (true_energy, true_position, true_direction)
    
    key, _ = jax.random.split(key)
    true_data = jax.lax.stop_gradient(data_simulator(true_params, detector_params, key, photon_data))
    
    hit_counts, hit_times_raw = true_data
    hit_times = hit_times_raw + TRUE_T0
    
    hit_mask = hit_counts > -999
    hit_detector_positions = detector_points[hit_mask]
    observed_times = hit_times[hit_mask]
    observed_counts = hit_counts[hit_mask]
    
    return {
        'event_idx': event_idx,
        'entry_idx': entry_idx,
        'true_energy': float(true_energy),
        'true_position': np.array(true_position),
        'true_direction': np.array(true_direction),
        'TRUE_T0': float(TRUE_T0),
        'true_data': true_data,
        'hit_detector_positions': hit_detector_positions,
        'observed_times': observed_times,
        'observed_counts': observed_counts,
        'n_hits': int(jnp.sum(hit_mask))
    }

print("Event generation function defined")

In [ ]:
# Multi-event optimization pipeline with Adam
verbosity = config.get('verbosity', {}).get('level', 2)
store_true_data = config.get('storage', {}).get('store_true_data', True)

print(f"Starting Adam optimization for {N_EVENTS} events...")
if verbosity >= 2:
    print(f"Storage mode: store_true_data = {store_true_data}")
    print("=" * 80)

# Storage for all event results
all_event_results = []

# Performance tracking arrays
energy_guess_errors = []
grid_position_errors = []
grid_t0_errors = []
cone_direction_errors = []
energy_scan_improvements = []
final_position_errors = []
final_direction_errors = []
final_t0_errors = []
final_energy_errors = []
final_combined_losses = []
final_vertex_losses = []
final_counts_losses = []
final_energy_losses = []
convergence_rates = []

# Generate random keys for all events
main_key = jax.random.PRNGKey(42)
event_keys = jax.random.split(main_key, N_EVENTS)

# Warm-up: Pre-compile all JIT functions with a dummy event
print("Pre-compiling JIT functions...")
warmup_event_data = generate_event_data(0, event_keys[0])
_ = run_complete_optimization_adam(
    initial_t0=0.0,
    hit_detector_positions=warmup_event_data['hit_detector_positions'],
    observed_times=warmup_event_data['observed_times'],
    observed_counts=warmup_event_data['observed_counts'],
    true_data=warmup_event_data['true_data'],
    true_energy=warmup_event_data['true_energy'],
    true_position=warmup_event_data['true_position'],
    true_direction=warmup_event_data['true_direction'],
    TRUE_T0=warmup_event_data['TRUE_T0'],
    max_iterations=1,
    verbosity=0
)
print("JIT compilation complete. Starting optimization...")

# Process each event
progress_bar = tqdm(range(N_EVENTS), desc="Processing events") if verbosity == 0 else range(N_EVENTS)
for event_idx in progress_bar:
    try:
        if verbosity >= 2:
            print(f"\n--- Processing Event {event_idx} ---")
        
        # Generate event data
        event_data = generate_event_data(event_idx, event_keys[event_idx])
        
        # Extract event parameters
        true_position = event_data['true_position']
        true_direction = event_data['true_direction']
        true_energy = event_data['true_energy']
        TRUE_T0 = event_data['TRUE_T0']
        true_data = event_data['true_data']
        hit_detector_positions = event_data['hit_detector_positions']
        observed_times = event_data['observed_times']
        observed_counts = event_data['observed_counts']
        
        # Convert true direction to spherical coordinates
        true_theta, true_phi = cartesian_to_spherical(true_direction)
        
        # Use t0_guess = 0.0 (we don't know the true t0)
        initial_t0 = 0.0
        
        if verbosity >= 2:
            print(f"  True position: {true_position}")
            print(f"  True direction: {true_direction}")
            print(f"  True energy: {true_energy:.1f}")
            print(f"  True t0: {TRUE_T0:.3f}")
            print(f"  Initial t0 guess: {initial_t0:.3f}")
        
        # Run complete optimization pipeline with Adam
        if verbosity >= 2:
            print("  Running Adam optimization...")
        results = run_complete_optimization_adam(
            initial_t0=initial_t0,
            hit_detector_positions=hit_detector_positions,
            observed_times=observed_times,
            observed_counts=observed_counts,
            true_data=true_data,
            true_energy=true_energy,
            true_position=true_position,
            true_direction=true_direction,
            TRUE_T0=TRUE_T0,
            max_iterations=MAX_ITERATIONS,
            verbosity=verbosity
        )
        
        if results is None:
            if verbosity >= 1:
                print(f"  ERROR: Optimization failed for event {event_idx}")
            continue
        
        # Calculate improvements from each stage
        energy_guess_error = results['energy_estimation']['energy_guess_error']
        grid_position_error = results['grid_position_search']['position_error']
        grid_t0_error = results['grid_position_search']['t0_error']
        
        # Calculate cone search direction error
        cone_direction = results['cone_direction_search']['best_direction']
        cone_cos_angle = np.clip(np.dot(cone_direction, true_direction), -1.0, 1.0)
        cone_direction_error = np.degrees(np.arccos(cone_cos_angle))
        
        energy_scan_improvement = results['energy_scan_search']['energy_improvement']
        
        # Create event_data copy for storage
        event_data_to_store = {
            'event_idx': event_data['event_idx'],
            'entry_idx': event_data['entry_idx'],
            'true_energy': event_data['true_energy'],
            'true_position': event_data['true_position'],
            'true_direction': event_data['true_direction'],
            'TRUE_T0': event_data['TRUE_T0'],
            'hit_detector_positions': event_data['hit_detector_positions'],
            'observed_times': event_data['observed_times'],
            'observed_counts': event_data['observed_counts'],
            'n_hits': event_data['n_hits']
        }
        
        if store_true_data:
            event_data_to_store['true_data'] = event_data['true_data']
        
        # Store results for this event
        event_result = {
            'event_data': event_data_to_store,
            'initial_t0': initial_t0,
            'true_theta': float(true_theta),
            'true_phi': float(true_phi),
            'energy_guess_error': energy_guess_error,
            'grid_position_error': grid_position_error,
            'grid_t0_error': grid_t0_error,
            'cone_direction_error': cone_direction_error,
            'energy_scan_improvement': energy_scan_improvement,
            'optimization_results': results
        }
        all_event_results.append(event_result)
        
        # Track performance metrics
        energy_guess_errors.append(energy_guess_error)
        grid_position_errors.append(grid_position_error)
        grid_t0_errors.append(grid_t0_error)
        cone_direction_errors.append(cone_direction_error)
        energy_scan_improvements.append(energy_scan_improvement)
        final_position_errors.append(results['final_position_error'])
        final_direction_errors.append(results['final_direction_error'])
        final_t0_errors.append(results['final_t0_error'])
        final_energy_errors.append(results['final_energy_error'])
        final_combined_losses.append(results['final_combined_loss'])
        final_vertex_losses.append(results['final_vertex_loss'])
        final_counts_losses.append(results['final_counts_loss'])
        final_energy_losses.append(results['final_energy_loss'])
        convergence_rates.append(1.0 if results['converged'] else 0.0)
        
        # Print results based on verbosity level
        if verbosity == 1:
            print(f"Event {event_idx}: pos_err={results['final_position_error']:.3f}m, "
                  f"dir_err={results['final_direction_error']:.1f}°, t0_err={results['final_t0_error']:.3f}, "
                  f"E_err={results['final_energy_error']:.1f}")
        elif verbosity >= 2:
            print(f"  Energy guess error: {energy_guess_error:.1f}")
            print(f"  Grid search - Position error: {grid_position_error:.3f}m, t0 error: {grid_t0_error:.3f}")
            print(f"  Cone direction error: {cone_direction_error:.1f}°")
            print(f"  Energy scan improvement: {energy_scan_improvement:.1f}")
            print(f"  Final errors - Position: {results['final_position_error']:.3f}m, "
                  f"Direction: {results['final_direction_error']:.1f}°, t0: {results['final_t0_error']:.3f}, "
                  f"Energy: {results['final_energy_error']:.1f}")
            print(f"  Converged: {results['converged']}, Iterations: {results['total_iterations']}")
            
    except Exception as e:
        if verbosity >= 1:
            print(f"Error processing event {event_idx}: {e}")
        if verbosity >= 2:
            import traceback
            traceback.print_exc()
        continue

if verbosity >= 1:
    print(f"\nCompleted processing {len(all_event_results)} events successfully")

In [ ]:
from tools.optimization.utils.functions import performance_summary

performance_summary(
    energy_guess_errors,
    grid_position_errors,
    cone_direction_errors,
    energy_scan_improvements,
    final_position_errors,
    final_direction_errors,
    final_t0_errors,
    final_energy_errors,
    final_combined_losses,
    final_vertex_losses,
    final_counts_losses,
    final_energy_losses,
    convergence_rates,
)

# Print additional t0 grid search statistics
print("\n" + "=" * 80)
print("4D Grid Search t0 Performance:")
print("=" * 80)
grid_t0_errors_arr = np.array(grid_t0_errors)
print(f"Mean t0 error:   {np.mean(grid_t0_errors_arr):.4f}")
print(f"Median t0 error: {np.median(grid_t0_errors_arr):.4f}")
print(f"Std t0 error:    {np.std(grid_t0_errors_arr):.4f}")
print(f"Min t0 error:    {np.min(grid_t0_errors_arr):.4f}")
print(f"Max t0 error:    {np.max(grid_t0_errors_arr):.4f}")

In [ ]:
# Save detailed results
output_dir = Path('../output')
output_dir.mkdir(parents=True, exist_ok=True)

energy_guess_errors = np.array(energy_guess_errors)
grid_position_errors = np.array(grid_position_errors)
grid_t0_errors = np.array(grid_t0_errors)
cone_direction_errors = np.array(cone_direction_errors)
energy_scan_improvements = np.array(energy_scan_improvements)
final_position_errors = np.array(final_position_errors)
final_direction_errors = np.array(final_direction_errors)
final_t0_errors = np.array(final_t0_errors)
final_energy_errors = np.array(final_energy_errors)
final_combined_losses = np.array(final_combined_losses)
final_vertex_losses = np.array(final_vertex_losses)
final_counts_losses = np.array(final_counts_losses)
final_energy_losses = np.array(final_energy_losses)
convergence_rates = np.array(convergence_rates)

results_summary = {
    'config': {
        'optimizer': 'Adam',
        'adam_learning_rate': ADAM_LEARNING_RATE,
        'adam_b1': ADAM_B1,
        'adam_b2': ADAM_B2,
        'adam_eps': ADAM_EPS,
        'n_events': N_EVENTS,
        'temperature': TEMPERATURE,
        'vertex_weight_scale': VERTEX_WEIGHT_SCALE,
        'counts_weight_scale': COUNTS_WEIGHT_SCALE,
        'energy_weight_scale': ENERGY_WEIGHT_SCALE,
        'pos_n_div': POS_N_DIV,
        'pos_levels': POS_LEVELS,
        'pos_fraction': POS_FRACTION,
        'pos_min_L': POS_MIN_L,
        't0_n_div': T0_N_DIV,
        't0_min': T0_MIN,
        't0_max': T0_MAX,
        'cone_levels': CONE_LEVELS,
        'cone_initial_div': CONE_INITIAL_DIV,
        'cone_max_angle_deg': CONE_MAX_ANGLE_DEG,
        'cone_reduction': CONE_REDUCTION,
        'energy_delta': ENERGY_DELTA,
        'energy_scan_steps': ENERGY_SCAN_STEPS,
        'detector_r': float(DETECTOR_R) if DETECTOR_R is not None else None,
        'detector_h': float(DETECTOR_H) if DETECTOR_H is not None else None,
        'detector_bounds': detector_bounds,
        'data_file': data_file,
        'detector_file': default_json_filename
    },
    'raw_data': {
        'energy_guess_errors': energy_guess_errors.tolist(),
        'grid_position_errors': grid_position_errors.tolist(),
        'grid_t0_errors': grid_t0_errors.tolist(),
        'cone_direction_errors': cone_direction_errors.tolist(),
        'energy_scan_improvements': energy_scan_improvements.tolist(),
        'final_position_errors': final_position_errors.tolist(),
        'final_direction_errors': final_direction_errors.tolist(),
        'final_t0_errors': final_t0_errors.tolist(),
        'final_energy_errors': final_energy_errors.tolist(),
        'final_combined_losses': final_combined_losses.tolist(),
        'final_vertex_losses': final_vertex_losses.tolist(),
        'final_counts_losses': final_counts_losses.tolist(),
        'final_energy_losses': final_energy_losses.tolist(),
        'convergence_rates': convergence_rates.tolist()
    },
    'all_event_results': all_event_results
}

# Save results
output_file = output_dir / f'single_ring_optimization_adam_{N_EVENTS}_results.pkl'
with open(output_file, 'wb') as f:
    pickle.dump(results_summary, f)

print(f"\n💾 Results saved:")
print(f"  - Complete results: {output_file}")